# 📏 Crack Detection Notebook: Precision SAM 2 + Skeletonization
## Detect (YOLO) → Segment (SAM 2) → Measure (Skeletonization)

---

**Updates:**
1.  **Strict BBox Masking:** Segmentations are now clipped to bounding boxes to ensure accuracy.
2.  **Pothole-Style Visuals:** Translucent solid masks with high-contrast skeletons.
3.  **Detailed Labels:** Shows "Cracks" and "Length: X.Xm" on the map clips.

**Output:** `detections.csv` and annotated images in `final_results.zip`.

## 1️⃣ Setup & Installation

In [ ]:
# @title 1️⃣ Install Dependencies
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip install -q -U ultralytics
!pip install -q git+https://github.com/facebookresearch/segment-anything-2.git
!pip install -q opencv-python pandas numpy tqdm scikit-image

# Download SAM 2 Checkpoint
!wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

print("✅ Setup Complete.")

In [ ]:
# @title 2️⃣ Configuration & File Paths
import os

VIDEO_PATH = "/content/video.mp4" # @param {type:"string"}
GPS_LOG_PATH = "/content/gps_log.csv" # @param {type:"string"}
DET_MODEL_PATH = "/content/best_cracks_rdd.pt" # @param {type:"string"}
CALIB_IMAGE_PATH = "/content/calibration.jpg" # @param {type:"string"}

FRAME_SKIP = 5 # @param {type:"integer"}
CONFIDENCE_THRESHOLD = 0.25 # @param {type:"number"}
IMG_SIZE = 640 # @param {type:"integer"}

USE_TRACKING = True # @param {type:"boolean"}
GENERATE_VIDEO = False # @param {type:"boolean"}
SAVE_IMAGES = True # @param {type:"boolean"}

ROAD_WIDTH_M = 3.5 # @param {type:"number"}
CAMERA_HEIGHT_M = 1.5 # @param {type:"number"}
MANUAL_CALIB_POINTS = [] # @param {type:"raw"}

print(f"✅ Config Loaded.")

## 2️⃣ Core Functions

In [ ]:
import cv2
import numpy as np
from skimage.morphology import skeletonize

def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0], rect[2] = pts[np.argmin(s)], pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1], rect[3] = pts[np.argmin(diff)], pts[np.argmax(diff)]
    return rect

def calculate_homography_and_scale(calibration_image_path, manual_points=None, road_width_m=3.5):
    if not os.path.exists(calibration_image_path): return None, None
    img = cv2.imread(calibration_image_path)
    if img is None: return None, None
    rect = None
    if manual_points and len(manual_points) == 4:
        rect = order_points(np.array(manual_points, dtype="float32"))
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        thresh = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)
        thresh = cv2.bitwise_not(thresh)
        cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = sorted(cnts, key=cv2.contourArea, reverse=True)
        for c in cnts:
            peri = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, 0.02 * peri, True)
            if len(approx) == 4:
                rect = order_points(approx.reshape(4, 2))
                print("✅ Auto-Detected A4.")
                break
    if rect is None: return None, None
    scale = 1000.0
    dst = np.array([[0,0], [0.21*scale, 0], [0.21*scale, 0.297*scale], [0, 0.297*scale]], dtype="float32")
    H = cv2.getPerspectiveTransform(rect, dst)
    return H, scale

def calculate_length_skeleton(mask, width, bbox=None, H=None, scale=1000.0):
    # --- STRICT BBOX CLIPPING ---
    # Creating a clean mask only inside the bounding box to avoid SAM leak
    if bbox is not None:
        x1, y1, x2, y2 = map(int, bbox)
        clean_mask = np.zeros_like(mask, dtype=bool)
        clean_mask[y1:y2, x1:x2] = mask[y1:y2, x1:x2] > 0.0
    else:
        clean_mask = mask > 0.0
    
    # Filter out small noise artifacts before skeletonizing
    # (Optional: can use cv2.morphologyEx but skimage works on bool)
    
    skeleton = skeletonize(clean_mask)
    pixel_length = np.sum(skeleton)
    
    if H is not None:
        length_m = pixel_length / scale
    else:
        ppm = (width * 0.8) / ROAD_WIDTH_M
        length_m = pixel_length / ppm
    return length_m, skeleton, clean_mask

## 3️⃣ Load Models

In [ ]:
from ultralytics import YOLO
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
yolo_model = YOLO(DET_MODEL_PATH) if os.path.exists(DET_MODEL_PATH) else None

sam2_checkpoint = "sam2_hiera_small.pt"
model_cfg = "sam2_hiera_s.yaml"
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
sam2_predictor = SAM2ImagePredictor(sam2_model)
print("✅ Models Loaded")

## 4️⃣ Run Inference (BBox-Restricted)

In [ ]:
import cv2
from tqdm import tqdm
import pandas as pd
import numpy as np
import shutil

H_matrix, H_scale = calculate_homography_and_scale(CALIB_IMAGE_PATH, MANUAL_CALIB_POINTS, ROAD_WIDTH_M)

gps_df = None
video_start_time = None
if os.path.exists(GPS_LOG_PATH):
    gps_df = pd.read_csv(GPS_LOG_PATH)
    gps_df.columns = gps_df.columns.str.strip().str.lower()
    t_col = next((c for c in gps_df.columns if 'time' in c or 'date' in c), None)
    if t_col:
        gps_df[t_col] = pd.to_datetime(gps_df[t_col], format='mixed', utc=True)
        gps_df = gps_df.sort_values(t_col).rename(columns={t_col: 'time'})
        video_start_time = gps_df['time'].iloc[0]

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
width, height = int(cap.get(3)), int(cap.get(4))
os.makedirs("results/frames", exist_ok=True)

final_detections = []
track_buffer = {} 
frame_idx = 0
pbar = tqdm(total=total_frames // FRAME_SKIP, desc="Analyzing Cracks")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    if frame_idx % FRAME_SKIP == 0:
        results = yolo_model.track(frame, persist=True, conf=CONFIDENCE_THRESHOLD, verbose=False)[0] if USE_TRACKING else yolo_model.predict(frame, conf=CONFIDENCE_THRESHOLD, verbose=False)[0]
        boxes = results.boxes.xyxy.cpu().numpy()
        track_ids = results.boxes.id.cpu().numpy().astype(int) if results.boxes.id is not None else [-1]*len(boxes)
        confs = results.boxes.conf.cpu().numpy()

        if len(boxes) > 0:
            sam2_predictor.set_image(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            masks, scores, _ = sam2_predictor.predict(point_coords=None, point_labels=None, box=boxes, multimask_output=False)
            if masks.ndim == 4: masks = masks[np.arange(masks.shape[0]), np.argmax(scores, axis=1)] if masks.shape[1] > 1 else masks.squeeze(1)

            annotated_frame = frame.copy()
            for i, (box, orig_mask, tid, conf) in enumerate(zip(boxes, masks, track_ids, confs)):
                # 1. Pothole-style Solid Mask + Skeleton (Restricted to Box)
                length_m, skeleton, clean_mask = calculate_length_skeleton(orig_mask, width, bbox=box, H=H_matrix, scale=H_scale)
                
                # Draw Solid Overlay (Cyan - 0.5 Alpha)
                overlay = annotated_frame.copy()
                overlay[clean_mask] = (255, 255, 0) # Cyan/Yellow-ish
                annotated_frame = cv2.addWeighted(overlay, 0.4, annotated_frame, 0.6, 0)
                
                # Draw High-Contrast Skeleton (Bright Blue)
                annotated_frame[skeleton] = (255, 0, 0)
                
                # Draw Bounding Box
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (255, 255, 0), 2)
                
                # Draw Labels at Upper Portion
                label_main = f"Cracks"
                label_sub = f"Length: {length_m:.2f}m"
                cv2.putText(annotated_frame, label_main, (x1, y1 - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
                cv2.putText(annotated_frame, label_sub, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

                # Metadata
                lat, lon = None, None
                if gps_df is not None and video_start_time is not None:
                    curr_t = video_start_time + pd.to_timedelta(frame_idx/fps, unit='s')
                    row = gps_df.iloc[(gps_df['time'] - curr_t).abs().idxmin()]
                    lat, lon = float(row['latitude']), float(row['longitude'])

                rec = {
                    'track_id': tid, 'length_m': round(length_m, 2), 'confidence': round(float(conf), 2), 
                    'frame_idx': frame_idx, 'latitude': lat, 'longitude': lon,
                    'image_path': f"crack_{tid}_{frame_idx}.jpg"
                }
                
                if USE_TRACKING and tid != -1:
                    if tid not in track_buffer: track_buffer[tid] = []
                    track_buffer[tid].append({'data': rec, 'img': annotated_frame})
                else:
                    if SAVE_IMAGES: cv2.imwrite(f"results/frames/{rec['image_path']}", annotated_frame)
                    final_detections.append(rec)

        pbar.update(1)
    frame_idx += 1

cap.release()
pbar.close()

if USE_TRACKING:
    for tid, history in track_buffer.items():
        if not history: continue
        avg_len = np.mean([h['data']['length_m'] for h in history])
        best = min(history, key=lambda x: abs(x['data']['length_m'] - avg_len))
        if SAVE_IMAGES: cv2.imwrite(f"results/frames/{best['data']['image_path']}", best['img'])
        final_detections.append(best['data'])

print(f"✅ Finished processing {len(final_detections)} cracks.")

## 5️⃣ Results Packaging & Download

In [ ]:
import shutil
import pandas as pd
from google.colab import files

if final_detections:
    df = pd.DataFrame(final_detections)
    # Standard CSV name for dashboard compatibility
    csv_p = "results/detections.csv"
    df.to_csv(csv_p, index=False)
    
    print("📦 Packaging findings...")
    shutil.make_archive("crack_results_v3", 'zip', "results")
    files.download("crack_results_v3.zip")
    print("🎉 Done! Download started.")
else:
    print("❌ No data to save.")